# GPU Tree Search Phase 1 — Results Analysis

**Comparison:** Before implementation (OpenACC hangs) vs After implementation (OpenACC completes)

## Key Result: NNI tree search now completes on GPU

In [1]:
import os
import re
import glob
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

%matplotlib agg

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['savefig.dpi'] = 150
matplotlib.rcParams['font.size'] = 12

# Paths — change these to point at your results
BEFORE_DIR = '/Users/u7826985/Projects/Nvidia/results/2026_03_21_treesearch_results_int'
AFTER_DIR = os.path.join(BEFORE_DIR, '2026_03_21_treesearch_results_after_impl')
SAVE_DIR = '/Users/u7826985/Projects/Nvidia/poc-gpu-likelihood-calculation_analysis/2026_03_21_treesearch_results_after_impl'

In [2]:
def parse_log(filepath):
    """Extract key metrics from an IQ-TREE log file."""
    results = {'filepath': filepath}
    if not os.path.exists(filepath):
        return None
    with open(filepath, 'r') as f:
        content = f.read()
    results['file_size'] = os.path.getsize(filepath)
    results['line_count'] = content.count('\n')
    patterns = {
        'best_score':       (r'BEST SCORE FOUND\s*:\s*([-\d.]+)', float),
        'tree_length':      (r'Total tree length.*?:\s*([\d.]+)', float),
        'tree_search_wall': (r'Wall-clock time used for tree search:\s*([\d.]+)', float),
        'tree_search_cpu':  (r'CPU time used for tree search:\s*([\d.]+)', float),
        'total_wall':       (r'Total wall-clock time used:\s*([\d.]+)', float),
        'total_cpu':        (r'Total CPU time used:\s*([\d.]+)', float),
    }
    for key, (pat, typ) in patterns.items():
        m = re.search(pat, content)
        results[key] = typ(m.group(1)) if m else None
    results['nni_rounds'] = len(re.findall(r'Doing NNI round', content))
    results['completed'] = 'BEST SCORE FOUND' in content
    return results


def parse_iqtree(filepath):
    """Extract model parameters from .iqtree report."""
    params = {}
    if not os.path.exists(filepath):
        return params
    with open(filepath, 'r') as f:
        content = f.read()
    m = re.search(r'Rate parameter R:', content)
    if m:
        section = content[m.start():m.start()+500]
        for pair in ['A-C', 'A-G', 'A-T', 'C-G', 'C-T', 'G-T']:
            m2 = re.search(rf'{pair}:\s*([\d.]+)', section)
            if m2: params[f'rate_{pair}'] = float(m2.group(1))
    for base in ['A', 'C', 'G', 'T']:
        m = re.search(rf'pi\({base}\)\s*=\s*([\d.]+)', content)
        if m: params[f'freq_{base}'] = float(m.group(1))
    m = re.search(r'Log-likelihood of the tree:\s*([-\d.]+)', content)
    if m: params['logl'] = float(m.group(1))
    m = re.search(r'Total tree length.*?:\s*([\d.]+)', content)
    if m: params['tree_length'] = float(m.group(1))
    return params


def extract_logl_trace(filepath):
    """Extract the sequence of 'Optimal log-likelihood:' values from a log."""
    values = []
    if not os.path.exists(filepath):
        return values
    with open(filepath, 'r') as f:
        for line in f:
            m = re.search(r'Optimal log-likelihood:\s*([-\d.]+)', line)
            if m:
                values.append(float(m.group(1)))
    return values


def find_logs(directory, workflow_pattern):
    """Dynamically discover log files matching a workflow pattern."""
    matches = glob.glob(os.path.join(directory, f'*_{workflow_pattern}_*.log'))
    full = [f for f in matches if 'fulltreesearch' in f]
    return full[0] if full else (matches[0] if matches else None)


def find_iqtree(directory, workflow_pattern):
    """Dynamically discover .iqtree files matching a workflow pattern."""
    matches = glob.glob(os.path.join(directory, f'*_{workflow_pattern}_*.iqtree'))
    full = [f for f in matches if 'fulltreesearch' in f]
    return full[0] if full else (matches[0] if matches else None)


print('Utility functions defined.')

Utility functions defined.


## 1. Discover and Parse Results

In [3]:
# Dynamically discover log files in BEFORE and AFTER directories
runs = {}
for label, directory in [('BEFORE', BEFORE_DIR), ('AFTER', AFTER_DIR)]:
    for wf in ['OPENACC', 'VANILA']:
        log_path = find_logs(directory, wf)
        if log_path:
            key = f'{label}_{wf}'
            runs[key] = parse_log(log_path)
            runs[key]['label'] = label
            runs[key]['workflow'] = wf
            print(f'{key}: {os.path.basename(log_path)}')
        else:
            print(f'{label}_{wf}: No log file found in {directory}')

print(f'\nTotal runs parsed: {len(runs)}')

BEFORE_OPENACC: output_test_fulltreesearch_DNA_GTR+R4_OPENACC_run1_tree_1_10_iqtree3_OPENACC_run1_tree_1_10_iqtree.log
BEFORE_VANILA: output_test_fulltreesearch_DNA_GTR+R4_VANILA_run1_tree_1_10_iqtree3_VANILA_run1_tree_1_10_iqtree.log
AFTER_OPENACC: output_test_fulltreesearch_DNA_GTR+R4_OPENACC_run1_tree_1_10_iqtree3_OPENACC_run1_tree_1_10_iqtree.log
AFTER_VANILA: output_test_fulltreesearch_DNA_GTR+R4_VANILA_run1_tree_1_10_iqtree3_VANILA_run1_tree_1_10_iqtree.log

Total runs parsed: 4


## 2. Before vs After: Completion Status

In [4]:
rows = []
for key in ['BEFORE_OPENACC', 'BEFORE_VANILA', 'AFTER_OPENACC', 'AFTER_VANILA']:
    r = runs.get(key)
    if r:
        rows.append({
            'Run': key.replace('_', ' '),
            'Completed': r['completed'],
            'Log Lines': r['line_count'],
            'NNI Rounds': r['nni_rounds'],
            'Best Score': r['best_score'],
        })

df_comp = pd.DataFrame(rows)
print('=' * 80)
print('COMPLETION STATUS: Before vs After Implementation')
print('=' * 80)
print(df_comp.to_string(index=False))

before_oa = runs.get('BEFORE_OPENACC', {})
after_oa = runs.get('AFTER_OPENACC', {})
print(f"\nBEFORE OpenACC: {'COMPLETED' if before_oa.get('completed') else 'HUNG/CRASHED'}")
print(f"AFTER  OpenACC: {'COMPLETED' if after_oa.get('completed') else 'HUNG/CRASHED'}")

COMPLETION STATUS: Before vs After Implementation
           Run  Completed  Log Lines  NNI Rounds  Best Score
BEFORE OPENACC      False       1422           1         NaN
 BEFORE VANILA       True     264750         103  -27.562937
 AFTER OPENACC       True     266489         103  -27.562937
  AFTER VANILA       True     264750         103  -27.562937

BEFORE OpenACC: HUNG/CRASHED
AFTER  OpenACC: COMPLETED


## 3. Correctness Verification (After Implementation)

In [5]:
oa = runs.get('AFTER_OPENACC', {})
va = runs.get('AFTER_VANILA', {})

if oa.get('best_score') is not None and va.get('best_score') is not None:
    logl_diff = abs(oa['best_score'] - va['best_score'])
    tl_diff = abs((oa.get('tree_length') or 0) - (va.get('tree_length') or 0))

    print('=' * 85)
    print('CORRECTNESS VERIFICATION: OpenACC vs VANILLA (After Implementation)')
    print('=' * 85)
    print(f"\n{'Metric':<30} {'OpenACC':>20} {'VANILLA':>20} {'Diff':>15}")
    print('-' * 85)
    print(f"{'Best log-likelihood':<30} {oa['best_score']:>20.10f} {va['best_score']:>20.10f} {logl_diff:>15.2e}")
    if oa.get('tree_length') and va.get('tree_length'):
        print(f"{'Total tree length':<30} {oa['tree_length']:>20.10f} {va['tree_length']:>20.10f} {tl_diff:>15.2e}")
    print(f"{'NNI rounds executed':<30} {oa['nni_rounds']:>20d} {va['nni_rounds']:>20d} {abs(oa['nni_rounds']-va['nni_rounds']):>15d}")
    
    print(f"\n{'=' * 85}")
    if logl_diff < 1e-4:
        print(f'VERDICT: CORRECTNESS VERIFIED - logL diff = {logl_diff:.2e} (within tolerance)')
    else:
        print(f'VERDICT: MISMATCH DETECTED - logL diff = {logl_diff:.2e}')
else:
    print('Cannot verify — AFTER runs missing or incomplete.')

CORRECTNESS VERIFICATION: OpenACC vs VANILLA (After Implementation)

Metric                                      OpenACC              VANILLA            Diff
-------------------------------------------------------------------------------------
Best log-likelihood                  -27.5629373673       -27.5629373702        2.90e-09
Total tree length                      0.8078039938         0.8078040434        4.96e-08
NNI rounds executed                             103                  103               0

VERDICT: CORRECTNESS VERIFIED - logL diff = 2.90e-09 (within tolerance)


## 4. Performance Comparison

In [6]:
if oa.get('total_wall') and va.get('total_wall'):
    print('=' * 80)
    print('PERFORMANCE COMPARISON (After Implementation)')
    print('=' * 80)
    print(f"\n{'Metric':<35} {'OpenACC':>15} {'VANILLA':>15} {'Ratio (OA/VA)':>15}")
    print('-' * 80)

    for label, key in [('Tree search wall-clock (s)', 'tree_search_wall'),
                       ('Tree search CPU (s)', 'tree_search_cpu'),
                       ('Total wall-clock (s)', 'total_wall'),
                       ('Total CPU (s)', 'total_cpu')]:
        oa_val = oa.get(key)
        va_val = va.get(key)
        if oa_val is not None and va_val is not None and va_val > 0:
            ratio = oa_val / va_val
            print(f'{label:<35} {oa_val:>15.4f} {va_val:>15.4f} {ratio:>14.2f}x')
        else:
            print(f'{label:<35} {str(oa_val):>15} {str(va_val):>15} {"N/A":>15}')

    ratio_total = oa['total_wall'] / va['total_wall']
    if ratio_total > 1:
        print(f'\nNote: GPU is {ratio_total:.1f}x slower on this dataset.')
    else:
        print(f'\nGPU is {1/ratio_total:.1f}x FASTER on this dataset.')

PERFORMANCE COMPARISON (After Implementation)

Metric                                      OpenACC         VANILLA   Ratio (OA/VA)
--------------------------------------------------------------------------------
Tree search wall-clock (s)                   0.5052          0.0837           6.03x
Tree search CPU (s)                          0.4499          0.0577           7.80x
Total wall-clock (s)                         4.5505          0.9601           4.74x
Total CPU (s)                                4.1253          0.6416           6.43x

Note: GPU is 4.7x slower on this dataset.


## 5. Visualization: Before vs After

In [7]:
plt.close('all')
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Plot 1: Completion Status ---
ax = axes[0]
run_keys = ['BEFORE_OPENACC', 'BEFORE_VANILA', 'AFTER_OPENACC', 'AFTER_VANILA']
categories = [k.replace('_', '\n') for k in run_keys]
completed = [runs.get(k, {}).get('completed', False) for k in run_keys]
bar_colors = ['#ff4444' if not c else '#44aa44' for c in completed]
bars = ax.bar(categories, [1]*len(run_keys), color=bar_colors, edgecolor='black', linewidth=1.5)
ax.set_ylim(0, 1.3)
ax.set_yticks([])
for bar, c in zip(bars, completed):
    ax.text(bar.get_x() + bar.get_width()/2, 0.5,
            'COMPLETED' if c else 'HUNG',
            ha='center', va='center', fontweight='bold', fontsize=13, color='white')
ax.set_title('Tree Search Completion Status', fontsize=14, fontweight='bold')

# --- Plot 2: Log-Likelihood Comparison ---
ax = axes[1]
score_data = []
score_labels = []
for key, lbl in [('BEFORE_VANILA', 'BEFORE\nVANILLA'), ('AFTER_VANILA', 'AFTER\nVANILLA'), ('AFTER_OPENACC', 'AFTER\nOpenACC')]:
    s = runs.get(key, {}).get('best_score')
    if s is not None:
        score_data.append(s)
        score_labels.append(lbl)

if score_data:
    score_colors = ['#6688cc', '#4477bb', '#ee8833'][:len(score_data)]
    bars = ax.bar(score_labels, score_data, color=score_colors, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Log-Likelihood')
    ax.set_title('Best Log-Likelihood Scores', fontsize=14, fontweight='bold')
    # Put labels above bars, not inside (avoids bbox explosion when values are nearly identical)
    for bar, score in zip(bars, score_data):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 0.98,
                f'{score:.4f}', ha='center', va='top', fontsize=9, fontweight='bold')
    # Safe ylim for negative log-likelihoods
    lo, hi = min(score_data), max(score_data)
    spread = abs(hi - lo)
    if spread < abs(lo) * 1e-6:
        # Values are essentially identical — show narrow range centered on the value
        center = (lo + hi) / 2
        margin = abs(center) * 0.01
        ax.set_ylim(center - margin, 0)
    else:
        margin = spread * 0.5
        ax.set_ylim(lo - margin, hi + margin)

# --- Plot 3: Log Lines (proxy for run progress) ---
ax = axes[2]
line_counts = [runs.get(k, {}).get('line_count', 0) for k in run_keys]
bar_colors2 = ['#ff6666', '#6688cc', '#ee8833', '#4477bb']
bars = ax.bar(categories, line_counts, color=bar_colors2, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Log File Lines')
ax.set_title('Log File Size (run progress)', fontsize=14, fontweight='bold')
for bar, lc in zip(bars, line_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{lc:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, max(line_counts) * 1.15 if line_counts else 1)

plt.subplots_adjust(wspace=0.35)
fig.savefig(os.path.join(SAVE_DIR, 'before_vs_after_comparison.png'), dpi=150)
plt.close(fig)
print('Figure saved.')

Figure saved.


## 6. Log-Likelihood Convergence Trace

In [8]:
# Extract convergence traces from AFTER logs
oa_log = find_logs(AFTER_DIR, 'OPENACC')
va_log = find_logs(AFTER_DIR, 'VANILA')

oa_trace = extract_logl_trace(oa_log) if oa_log else []
va_trace = extract_logl_trace(va_log) if va_log else []

print(f'OpenACC: {len(oa_trace)} optimal log-likelihood evaluations')
print(f'VANILLA: {len(va_trace)} optimal log-likelihood evaluations')

min_len = min(len(oa_trace), len(va_trace))
if min_len > 0:
    diffs = [abs(oa_trace[i] - va_trace[i]) for i in range(min_len)]
    print(f'\nFirst {min(10, min_len)} evaluations:')
    print(f'{"Step":<6} {"OpenACC":>20} {"VANILLA":>20} {"Diff":>15}')
    print('-' * 61)
    for i in range(min(10, min_len)):
        print(f'{i:<6} {oa_trace[i]:>20.10f} {va_trace[i]:>20.10f} {diffs[i]:>15.2e}')
    print(f'\nMax diff across all {min_len} steps: {max(diffs):.2e}')
    print(f'Mean diff: {np.mean(diffs):.2e}')

    # Downsample if too many points
    max_pts = 1000
    if min_len > max_pts:
        stride = min_len // max_pts
        idx = list(range(0, min_len, stride))
    else:
        idx = list(range(min_len))
    oa_plot = [oa_trace[i] for i in idx]
    va_plot = [va_trace[i] for i in idx]
    diffs_plot = [diffs[i] for i in idx]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    ax1.plot(idx, oa_plot, '-', linewidth=0.8, alpha=0.7, label='OpenACC', color='#ee8833')
    ax1.plot(idx, va_plot, '-', linewidth=0.8, alpha=0.7, label='VANILLA', color='#4477bb')
    ax1.set_xlabel('Evaluation Step')
    ax1.set_ylabel('Log-Likelihood')
    ax1.set_title('Log-Likelihood Trace (full search)')
    ax1.legend()

    nonzero_diffs = [d for d in diffs_plot if d > 0]
    if nonzero_diffs:
        ax2.plot(idx, diffs_plot, '-', color='red', linewidth=0.8)
        ax2.set_xlabel('Evaluation Step')
        ax2.set_ylabel('|OpenACC - VANILLA|')
        ax2.set_title('Absolute Difference per Step')
        ax2.set_yscale('log')
    else:
        ax2.text(0.5, 0.5, 'All differences = 0', ha='center', va='center', transform=ax2.transAxes, fontsize=14)
        ax2.set_title('Absolute Difference per Step')

    fig.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, 'convergence_trace.png'), dpi=150)
    plt.close(fig)
    print('Figure saved.')
else:
    print('No convergence trace data available.')

OpenACC: 236 optimal log-likelihood evaluations
VANILLA: 236 optimal log-likelihood evaluations

First 10 evaluations:
Step                OpenACC              VANILLA            Diff
-------------------------------------------------------------
0            -28.0710502020       -28.0710502717        6.97e-08
1            -27.8354570427       -27.8354567614        2.81e-07
2            -36.9427551661       -36.9427551661        0.00e+00
3            -36.9427574084       -36.9427574084        0.00e+00
4            -36.9439505249       -36.9439505249        0.00e+00
5            -37.0560937217       -37.0560936341        8.76e-08
6            -36.9428026498       -36.9428026498        0.00e+00
7            -36.9427553206       -36.9427553206        0.00e+00
8            -37.0251637947       -37.0251637947        0.00e+00
9            -36.9795479236       -36.9795479236        0.00e+00

Max diff across all 236 steps: 3.33e-05
Mean diff: 6.61e-07


Figure saved.


## 7. Model Parameter Comparison

In [9]:
oa_iqtree = find_iqtree(AFTER_DIR, 'OPENACC')
va_iqtree = find_iqtree(AFTER_DIR, 'VANILA')

oa_params = parse_iqtree(oa_iqtree) if oa_iqtree else {}
va_params = parse_iqtree(va_iqtree) if va_iqtree else {}

if oa_params and va_params:
    print('=' * 80)
    print('MODEL PARAMETER COMPARISON')
    print('=' * 80)
    print(f"\n{'Parameter':<20} {'OpenACC':>15} {'VANILLA':>15} {'Diff':>15}")
    print('-' * 65)
    all_keys = sorted(set(list(oa_params.keys()) + list(va_params.keys())))
    for key in all_keys:
        oa_val = oa_params.get(key)
        va_val = va_params.get(key)
        if oa_val is not None and va_val is not None:
            diff = abs(oa_val - va_val)
            print(f'{key:<20} {oa_val:>15.6f} {va_val:>15.6f} {diff:>15.2e}')
        else:
            print(f'{key:<20} {str(oa_val):>15} {str(va_val):>15} {"N/A":>15}')
else:
    print('No .iqtree report files found for parameter comparison.')

MODEL PARAMETER COMPARISON

Parameter                    OpenACC         VANILLA            Diff
-----------------------------------------------------------------
freq_A                      0.275000        0.275000        0.00e+00
freq_C                      0.325000        0.325000        0.00e+00
freq_G                      0.000000        0.000000        0.00e+00
freq_T                      0.400000        0.400000        0.00e+00
logl                      -27.562937      -27.562937        0.00e+00
rate_A-C                    0.000100        0.000100        0.00e+00
rate_A-G                    0.128600        0.128600        0.00e+00
rate_A-T                    1.000000        1.000000        0.00e+00
rate_C-G                    0.000100        0.000100        0.00e+00
rate_C-T                    0.128600        0.128600        0.00e+00
rate_G-T                    1.000000        1.000000        0.00e+00
tree_length                 0.807804        0.807804        0.00e+00


## 8. Summary

In [10]:
print('=' * 90)
print('PHASE 1 RESULTS SUMMARY')
print('=' * 90)

def safe_fmt(val, fmt='.10f'):
    return f'{val:{fmt}}' if val is not None else 'N/A'

before_oa = runs.get('BEFORE_OPENACC', {})
after_oa = runs.get('AFTER_OPENACC', {})
after_va = runs.get('AFTER_VANILA', {})

logl_diff_str = 'N/A'
if after_oa.get('best_score') is not None and after_va.get('best_score') is not None:
    logl_diff_str = f"{abs(after_oa['best_score'] - after_va['best_score']):.2e}"

tl_diff_str = 'N/A'
if after_oa.get('tree_length') is not None and after_va.get('tree_length') is not None:
    tl_diff_str = f"{abs(after_oa['tree_length'] - after_va['tree_length']):.2e}"

summary_data = [
    ('NNI Tree Search', 
     'HUNG' if not before_oa.get('completed') else 'COMPLETED',
     'COMPLETED' if after_oa.get('completed') else 'FAILED',
     'COMPLETED' if after_va.get('completed') else 'FAILED'),
    ('Best Log-Likelihood', 
     safe_fmt(before_oa.get('best_score')),
     safe_fmt(after_oa.get('best_score')),
     safe_fmt(after_va.get('best_score'))),
    ('logL Diff (OA vs VA)',
     'N/A (hung)', logl_diff_str, '(reference)'),
    ('Total Tree Length',
     safe_fmt(before_oa.get('tree_length')),
     safe_fmt(after_oa.get('tree_length')),
     safe_fmt(after_va.get('tree_length'))),
    ('Tree Length Diff',
     'N/A', tl_diff_str, '(reference)'),
    ('NNI Rounds',
     str(before_oa.get('nni_rounds', 'N/A')),
     str(after_oa.get('nni_rounds', 'N/A')),
     str(after_va.get('nni_rounds', 'N/A'))),
    ('Tree Search Wall (s)',
     safe_fmt(before_oa.get('tree_search_wall'), '.4f'),
     safe_fmt(after_oa.get('tree_search_wall'), '.4f'),
     safe_fmt(after_va.get('tree_search_wall'), '.4f')),
    ('Total Wall (s)',
     safe_fmt(before_oa.get('total_wall'), '.4f'),
     safe_fmt(after_oa.get('total_wall'), '.4f'),
     safe_fmt(after_va.get('total_wall'), '.4f')),
]

summary_df = pd.DataFrame(summary_data, columns=['Metric', 'Before (OpenACC)', 'After (OpenACC)', 'After (VANILLA)'])
print(summary_df.to_string(index=False))

print('\n' + '=' * 90)
print('CONCLUSION:')
print(f'  1. Phase 1 fix: OpenACC {"COMPLETES" if after_oa.get("completed") else "STILL FAILS"} (was {"hanging" if not before_oa.get("completed") else "working"} before)')
if after_oa.get('best_score') and after_va.get('best_score'):
    print(f'  2. logL diff = {abs(after_oa["best_score"] - after_va["best_score"]):.2e}')
if after_oa.get('total_wall') and after_va.get('total_wall'):
    ratio = after_oa['total_wall'] / after_va['total_wall']
    if ratio > 1:
        print(f'  3. GPU is {ratio:.1f}x slower (expected on small datasets — GPU overhead dominates)')
    else:
        print(f'  3. GPU is {1/ratio:.1f}x faster')
print('=' * 90)

PHASE 1 RESULTS SUMMARY
              Metric Before (OpenACC) After (OpenACC) After (VANILLA)
     NNI Tree Search             HUNG       COMPLETED       COMPLETED
 Best Log-Likelihood              N/A  -27.5629373673  -27.5629373702
logL Diff (OA vs VA)       N/A (hung)        2.90e-09     (reference)
   Total Tree Length              N/A    0.8078039938    0.8078040434
    Tree Length Diff              N/A        4.96e-08     (reference)
          NNI Rounds                1             103             103
Tree Search Wall (s)              N/A          0.5052          0.0837
      Total Wall (s)              N/A          4.5505          0.9601

CONCLUSION:
  1. Phase 1 fix: OpenACC COMPLETES (was hanging before)
  2. logL diff = 2.90e-09
  3. GPU is 4.7x slower (expected on small datasets — GPU overhead dominates)
